In [ ]:

import sys
import re
import csv
from bs4 import BeautifulSoup

# LRP types that indicate a road intersection
INTERSECTION_TYPES = {"cross road", "side road", "junction", "t-junction", "intersection"}

# Road code pattern: one or more letters followed by digits (e.g. N1, R110, Z1101, N105)
ROAD_CODE_RE = re.compile(r"\b([A-Z]\d+)\b")


def parse_file(filepath):
    """Parse an RMMS lrps.htm file and return list of LRP dicts."""
    with open(filepath, "r", encoding="iso-8859-1") as f:
        soup = BeautifulSoup(f, "html.parser")

    # The data table is the largest table with LRP rows
    tables = soup.find_all("table")
    if not tables:
        raise ValueError(f"No HTML tables found in {filepath}. Is this a valid RMMS lrps.htm?")
    data_table = max(tables, key=lambda t: len(t.find_all("tr")))
    rows = data_table.find_all("tr")

    # Detect road name from title row
    road_name = "Unknown"
    title_match = re.search(r"LRPs\s+of\s+(\w+)", rows[0].get_text())
    if title_match:
        road_name = title_match.group(1)

    records = []
    for row in rows[2:]:  # skip title + header rows
        cells = row.find_all("td")
        texts = [c.get_text(strip=True) for c in cells]
        # Expect: ['', lrp_id, chainage, lrp_type, description, lat, lon, '']
        if len(texts) < 7:
            continue
        _, lrp_id, chainage, lrp_type, description, lat, lon, *_ = texts
        if not lrp_id:
            continue
        records.append({
            "road":        road_name,
            "lrp_id":      lrp_id.strip(),
            "chainage_km": chainage.strip(),
            "lrp_type":    lrp_type.strip(),
            "description": description.strip(),
            "latitude":    lat.strip(),
            "longitude":   lon.strip(),
            "source_file": filepath,
        })

    return records


def extract_intersections(records):
    """Filter records that intersect with N-roads only."""
    results = []
    for rec in records:
        #change all strings to lowercase
        lrp_type_lower = rec["lrp_type"].lower()
        desc_lower = rec["description"].lower()

        is_intersection = any(
            t in lrp_type_lower or t in desc_lower
            for t in INTERSECTION_TYPES
        )
        if not is_intersection:
            continue

        # Only keep road codes starting with N
        road_codes = [r for r in ROAD_CODE_RE.findall(rec["description"]) if r.startswith("N")]
        if not road_codes:
            continue

        #with dictionary unpacking, create a new record with joined road codes
        results.append({
            **rec,
            "intersecting_roads": ", ".join(road_codes),
        })
    return results


def print_results(intersections):
    if not intersections:
        print("  No intersections found.")
        return

    # Group by road. .setdefault() groups intersections by parents road name
    by_road = {}
    for rec in intersections:
        by_road.setdefault(rec["road"], []).append(rec)

    #iterate through the grouped data and print details
    for road, recs in by_road.items():
        print(f"\n{'='*72}")
        print(f"  Road: {road}  ({len(recs)} intersection(s))")
        print(f"{'='*72}")
        for rec in recs:
            print(f"  LRP ID     : {rec['lrp_id']}")
            print(f"  Chainage   : {rec['chainage_km']} km")
            print(f"  Type       : {rec['lrp_type']}")
            print(f"  Description: {rec['description']}")
            print(f"  Intersects : {rec['intersecting_roads']}")
            print(f"  Coordinates: {rec['latitude']}, {rec['longitude']}")
            print()

# use csv_DictWriter and .writeheader() to set up the csv file
def save_csv(intersections, path):
    if not intersections:
        return
    fields = ["road", "lrp_id", "chainage_km", "lrp_type", "description",
              "intersecting_roads", "latitude", "longitude", "source_file"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(intersections)
    print(f"\nCSV saved → {path}")

#define default files
JUPYTER_FILES = [
    "N1.lrps.htm",
    "N2.lrps.htm",
]

is_jupyter = any("jupyter" in a or "ipykernel" in a or a.endswith(".json")
                 for a in sys.argv)

files = JUPYTER_FILES if is_jupyter else sys.argv[1:]

if not files:
    print("Usage: python extract_intersections.py N1_lrps.htm N2_lrps.htm ...")
    sys.exit(1)

#loop through files, parse files with parse_file() and use .extend() to merge the record lists
all_records = []
for fp in files:
    print(f"Parsing: {fp}")
    try:
        recs = parse_file(fp)
        print(f"  → {len(recs)} LRP records loaded")
        all_records.extend(recs)
    except FileNotFoundError:
        print(f"  ERROR: File not found: {fp}")

#final execution, display and save to csv
print(f"\nTotal records: {len(all_records)}")

intersections = extract_intersections(all_records)
print(f"Total intersections found: {len(intersections)}")
print_results(intersections)
save_csv(intersections, "intersections.csv")

In [15]:

import sys
import csv
import pandas as pd
from math import radians, sin, cos, sqrt, atan2

#define default files
JUPYTER_FILES = {
    "reference": "../preprocessing/processed_data.csv",
    "generated": "intersections.csv",
}

#haversine formula to convert degrees to meters
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

#parsing the reference csv to do raw line processing
def load_reference(filepath):
    rows = []
    with open(filepath, newline="", encoding="utf-8") as f:
        for line in csv.reader(f):
            if not line or not line[0].strip():
                continue
            parts = line[0].strip().split("/")
            if len(parts) != 2:
                continue
            try:
                lat = float(line[5])
                lon = float(line[6])
            except (IndexError, ValueError):
                continue
            rows.append({
                "road_pair": tuple(sorted(parts)),
                "ref_lat":   lat,
                "ref_lon":   lon,
            })
    return rows

#loading generated csv file using Dictreader to access the column names
def load_generated(filepath):
    rows = []
    with open(filepath, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            for iroad in row["intersecting_roads"].split(", "):
                rows.append({
                    "road_pair": tuple(sorted([row["road"].strip(), iroad.strip()])),
                    "gen_lat":   float(row["latitude"]),
                    "gen_lon":   float(row["longitude"]),
                    "lrp_id":    row["lrp_id"],
                })
    return rows

#comparing the reference points to the closest generated LRP point
##creating a dictionary using set.default() to group candidates by road pair
def compare(ref_rows, gen_rows):
    gen_lookup = {}
    for g in gen_rows:
        gen_lookup.setdefault(g["road_pair"], []).append(g)

    results = []
    for ref in ref_rows:
        pair = ref["road_pair"]
        road_name = "/".join(pair)
        candidates = gen_lookup.get(pair, [])

        if candidates:
            # Pick the candidate closest to the reference point
            gen = min(
                candidates,
                key=lambda g: haversine_m(ref["ref_lat"], ref["ref_lon"], g["gen_lat"], g["gen_lon"])
            )
            results.append({
                "road_intersection": road_name,
                "lrp_id":           gen["lrp_id"],
                "ref_lat":          ref["ref_lat"],
                "ref_lon":          ref["ref_lon"],
                "gen_lat":          gen["gen_lat"],
                "gen_lon":          gen["gen_lon"],
                "status":           "FOUND",
            })
        else:
            #handles the cases where no match was found
            results.append({
                "road_intersection": road_name,
                "lrp_id":           None,
                "ref_lat":          ref["ref_lat"],
                "ref_lon":          ref["ref_lon"],
                "gen_lat":          None,
                "gen_lon":          None,
                "status":           "MISSING",
            })

    return pd.DataFrame(results)

#use pandas boolean indexing and .sum() method to count match statuses
def print_report(df):
    total   = len(df)
    found   = (df["status"] == "FOUND").sum()
    missing = (df["status"] == "MISSING").sum()

    print(f"\n{'='*72}")
    print(f"  Summary: {found}/{total} matched, {missing}/{total} missing")
    print(f"{'='*72}\n")

    print("Full comparison table:")
    print(df.to_string(index=False))

    #filter the df to display only the missing intersections
    if missing > 0:
        print(f"\n❌ Missing intersections:")
        print(df[df["status"] == "MISSING"][["road_intersection", "ref_lat", "ref_lon"]].to_string(index=False))


# ── Entry point ───────────────────────────────────────────────────────────────
is_jupyter = any(
    "jupyter" in a or "ipykernel" in a or a.endswith(".json")
    for a in sys.argv
)

if is_jupyter:
    ref_file = JUPYTER_FILES["reference"]
    gen_file = JUPYTER_FILES["generated"]
elif len(sys.argv) == 3:
    ref_file = sys.argv[1]
    gen_file = sys.argv[2]
else:
    print("Usage: python compare_intersections.py reference.csv intersections.csv")
    sys.exit(1)

print(f"Loading reference file : {ref_file}")
ref_rows = load_reference(ref_file)
print(f"  → {len(ref_rows)} reference rows loaded")

print(f"Loading generated file : {gen_file}")
gen_rows = load_generated(gen_file)
print(f"  → {len(gen_rows)} generated rows loaded")

df = compare(ref_rows, gen_rows)
print_report(df)

Loading reference file : ../preprocessing/processed_data.csv
  → 14 reference rows loaded
Loading generated file : intersections.csv
  → 19 generated rows loaded

  Summary: 14/14 matched, 0/14 missing

Full comparison table:
road_intersection  lrp_id   ref_lat   ref_lon   gen_lat   gen_lon status
            N1/N2 LRP009a 23.706083 90.521527 23.706083 90.521527  FOUND
          N1/N102 LRP084a 23.478972 91.118166 23.478972 91.118166  FOUND
          N1/N102 LRP084a 23.478944 91.117722 23.478972 91.118166  FOUND
          N102/N2 LRP086a 24.050833 91.114444 24.050833 91.114444  FOUND
          N1/N104 LRP148a 23.009556 91.381360 23.009556 91.381360  FOUND
          N1/N105 LRP012c 23.690416 90.546583 23.690416 90.546583  FOUND
          N105/N2 LRP012a 23.785389 90.568888 23.785333 90.568555  FOUND
          N2/N204 LRP117b 24.147861 91.346444 24.147861 91.346444  FOUND
          N2/N204 LRP117b 24.149889 91.347444 24.147861 91.346444  FOUND
          N2/N204 LRP142c 24.267694 91.47688

In [1]:

import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

COMPARISON_FILE = "intersection_comparison.csv"


def haversine_m(lat1, lon1, lat2, lon2):
    """Return distance in metres between two lat/lon points."""
    R = 6_371_000  # Earth radius in metres
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


df = pd.read_csv(COMPARISON_FILE)

# Only compare rows where both coordinate sets are present
df_found = df[df["status"] == "FOUND"].copy()

df_found["distance_m"] = df_found.apply(
    lambda r: haversine_m(r["ref_lat"], r["ref_lon"], r["gen_lat"], r["gen_lon"]),
    axis=1
)

# ── Statistics ────────────────────────────────────────────────────────────────
stats = df_found["distance_m"].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max",
)
# print the descriptive statistics for a quick data quality check
print(f"\n{'='*55}")
print(f"  Distance statistics (ref vs generated coordinates)")
print(f"{'='*55}")
print(f"  Count   : {int(stats['count'])}")
print(f"  Mean    : {stats['mean']:.1f} m")
print(f"  Median  : {stats['median']:.1f} m")
print(f"  Std Dev : {stats['std']:.1f} m")
print(f"  Min     : {stats['min']:.1f} m")
print(f"  Max     : {stats['max']:.1f} m")
print(f"{'='*55}\n")

#use .sortvalues() to rank the distances and .to_string to display as strings
print("Per-intersection distances:")
print(
    df_found[["road_intersection", "lrp_id", "ref_lat", "ref_lon",
              "gen_lat", "gen_lon", "distance_m"]]
    .sort_values("distance_m", ascending=False)
    .to_string(index=False)
)

FileNotFoundError: [Errno 2] No such file or directory: 'intersection_comparison.csv'